# 🧪 Lab 2 : Gouvernance GCP, IAM, Tarification & Garde-fous FinOps

*(Inspiré des labs officiels Google Cloud Skills Boost : "IAM Options & Roles" et "Google Cloud Billing & Cost Management")*

Dans ce lab pratique, vous allez explorer les principes fondamentaux de la gouvernance sur **Google Cloud Platform (GCP)** : la hiérarchie des ressources, la gestion des accès via **IAM (Identity and Access Management)** et les mécanismes de protection financière (**FinOps**, **quotas** et **garde-fous par requête**).

### 🎯 Objectifs :
1. **Inspecter l'organisation GCP** et la configuration du projet actif via la CLI `gcloud`.
2. **Auditer la stratégie IAM** et expérimenter le **Principe du Moindre Privilège** (*Least Privilege*).
3. **Vérifier la Résidence des Données (RGPD)** et comprendre l'impact de la colocalisation entre Cloud Storage et BigQuery.
4. **Tester les garde-fous de facturation BigQuery (`maximum_bytes_billed`)** pour bloquer automatiquement toute requête trop coûteuse avant exécution (coût $0).
5. **Examiner les quotas et budgets de sécurité** appliqués sur l'environnement de cours.

---


## 0. Configuration de l'environnement CLI (Cloud Shell / VS Code Terminal)

Exécutez la cellule de code ci-dessous pour identifier votre identité active et vérifier les propriétés du projet GCP du cours :


In [ ]:
%%bash
# 1. Vérifier votre compte utilisateur Google Cloud authentifié
gcloud auth list

# 2. Récupérer l'ID du projet GCP actif
export PROJECT_ID=$(gcloud config get-value project)
echo "Projet GCP actif : $PROJECT_ID"

# 3. Afficher les métadonnées de base du projet
gcloud projects describe $PROJECT_ID --format="table(projectId, name, projectNumber, lifecycleState)"


> 🧠 **Rappel du Cours - Hiérarchie des Ressources** :
> GCP organise les ressources selon la hiérarchie suivante :
> `Organisation` ➔ `Dossiers (Folders)` ➔ `Projet` ➔ `Ressources (BigQuery, GCS, IAM)`.
> **Toute ressource doit impérativement appartenir à un Projet.** C'est au niveau du Projet que la facturation est rattachée.

---


## 1. Audit IAM & Expérimentation du Moindre Privilège (Least Privilege)

Le composant **IAM** contrôle **"Qui"** (Identité / Compte utilisateur) a le droit d'effectuer **"Quelle action"** (Rôle / Permissions) sur **"Quelle ressource"**.

### Étape 1.1 : Inspecter les rôles attribués à votre compte via la CLI

Exécutez la cellule ci-dessous pour lister la politique IAM attribuée à votre compte sur le projet :

### Étape 1.2 : Tester la sécurité dans la Console Graphique GCP (UI)

1. Rendez-vous dans le menu principal (☰) ➔ **IAM & Admin** ➔ **IAM**.
2. Dans la barre de recherche des membres, filtrez avec votre adresse e-mail d'étudiant.
3. Observez vos rôles attribués (ex: `BigQuery User`, `Viewer`, etc.).
4. **Test d'élévation de privilèges** (Sécurité) :
   - Cliquez sur l'icône **Crayon** (Modifier le membre) à côté de votre nom ou sur **Accorder l'accès** (*Grant Access*) en haut de la page.
   - Essayez de vous attribuer le rôle **Propriétaire** (*Owner*) ou **Administrateur IAM** (*IAM Admin*).
   - Cliquez sur **Enregistrer**.

🔍 **Observation attendue** :
Un message d'erreur rouge s'affiche : *« Permission requise resourcemanager.projects.setIamPolicy manquante »*.
👉 **Pourquoi ?** Votre compte est restreint selon le **Principe du Moindre Privilège** (*Least Privilege*) : vous disposez des accès nécessaires pour exécuter des requêtes analytiques et créer des datasets, mais pas pour modifier les règles de sécurité du projet.

---


## 2. Conformité & Résidence des Données (Data Residency & RGPD)

Sur Google Cloud, chaque ressource de stockage (Dataset BigQuery ou Bucket Cloud Storage) réside dans une localisation géographique précise :
- **Régionale** (ex: `europe-west9` pour Paris, `europe-west1` pour la Belgique).
- **Multi-régionale** (ex: `EU` pour l'Union Européenne, `US` pour les États-Unis).

> ⚠️ **Règle d'or de colocalisation** : BigQuery ne peut **JAMAIS** effectuer de requêtes d'ingestion ou de jointures entre des ressources situées dans des régions géographiques incompatibles (ex: lire un bucket GCS situé aux US depuis un dataset BigQuery situé en EU).

### Étape 2.1 : Tester l'incompatibilité de région dans BigQuery

1. Ouvrez la **Console BigQuery** : `console.cloud.google.com/bigquery`.
2. Essayez de créer une table externe pointant sur un bucket public hébergé aux États-Unis (`US`) dans un dataset configuré en `EU` :

In [ ]:
-- Tentative de création d'une table externe US dans un schéma EU
CREATE OR REPLACE EXTERNAL TABLE `bronze_landing.ext_us_sample` -- Schéma EU créé au lab précédent
OPTIONS (
  format = 'CSV',
  uris = ['gs://cloud-samples-data/bigquery/us-states/us-states.csv']
);

In [ ]:

-- 3. Exécutez une requête sur cette table :
SELECT * FROM `bronze_landing.ext_us_sample` LIMIT 10;


🔍 **Observation attendue** :
BigQuery renvoie une erreur d'emplacement : *« Cannot read source URI in location US from dataset in location EU »*.
👉 Pour corriger cela, le bucket GCS et le dataset BigQuery doivent impérativement se trouver dans la même zone géographique (`EU`).

---


## 3. Garde-fous FinOps & Contrôle Strict des Coûts (`maximum_bytes_billed`)

Sur BigQuery, le modèle tarifaire à la requête (*On-Demand*) gère les coûts au volume scanné (~6$ par Téraoctet scanné).
Afin d'empêcher l'exécution accidentelle de requêtes gigantesques (ex: scanner plusieurs Terabytes par erreur avec un `SELECT *`), BigQuery propose une option de sécurité fondamentale : **`maximum_bytes_billed`**.

### Comment ça marche ?
Lorsque vous définissez `maximum_bytes_billed = X` :
1. BigQuery effectue un **Dry Run** automatique et gratuit avant d'exécuter la requête.
2. Si l'estimation scannée dépasse la valeur $X$, BigQuery **annule immédiatement la requête**.
3. **Coût facturé = 0,00 €**.

--- 

### Étape 3.1 : Tester le Garde-fou via la CLI `bq`

Exécutez la cellule ci-dessous pour lancer une requête sur la table volumineuse des dépôts GitHub (`bigquery-public-data.samples.github_nested`, ~2.5 Go par colonne), en fixant un garde-fou strict à **10 Mo** (`10000000` octets) :

In [ ]:
# Exécution avec un garde-fou strict de 10 Mo (doit rejeter l'exécution sans facturation)
bq query \
  --use_legacy_sql=false \
  --maximum_bytes_billed=10000000 \
  "SELECT repository.name, COUNT(1) FROM \`bigquery-public-data.samples.github_nested\` GROUP BY 1"


🔍 **Observation attendue dans la sortie de cellule** :
```text
Error in query string: Error processing job '...': Query exceeded limit for bytes billed: 10000000. Provided 10000000, total scannable bytes were 2542109811.
```
👉 **Explication** : Le serveur BigQuery a stoppé le job **avant même qu'il ne démarre**. Aucune ressource CPU n'a été consommée, aucun centime n'a été dépensé !

--- 

### Étape 3.2 : Configurer le Garde-fou dans la Console Web BigQuery (UI)

Vous pouvez activer ce garde-fou directement dans votre interface de requête web :

1. Dans l'éditeur de requêtes de la console BigQuery, cliquez sur **Paramètres** (*More / Query settings*) en haut à droite du panneau de saisie.
2. Faites défiler jusqu'à la section **Paramètres avancés** (*Advanced options*).
3. Cochez / Saisissez dans **Octets facturés maximum** (*Maximum bytes billed*) : `1000000000` (soit 1 Go).
4. Cliquez sur **Enregistrer**.

---


## 4. Exercice Pratique : Auditer les Quotas & la Protection FinOps du Cours 🏋️

### Énoncé :

1. En vous appuyant sur la table `bigquery-public-data.wikipedia.pageviews_2020` (table de plusieurs dizaines de Go) :
   - Rédigez une requête calculant le nombre total de vues pour le projet `'fr'` sur le mois de Janvier 2020.
   - **Avant d'exécuter** : Observez l'estimation du Dry Run (coche verte dans la console UI).
   - Si le volume scanné dépasse votre garde-fou (ex: 1 Go), ajustez votre clause `WHERE` (par exemple en filtrant sur une date précise comme `datehour >= '2020-01-01' AND datehour < '2020-01-02'`) pour faire passer le volume sous le seuil maximal de 1 Go autorisé !

2. **Découverte des Quotas Système GCP** :
   - Allez dans le menu GCP (☰) ➔ **IAM & Admin** ➔ **Quotas et limites système**.
   - Dans le filtre de recherche, tapez `BigQuery API`.
   - Observez le quota **Query usage per day per user** (fixé sur l'environnement de cours à 33 Go / jour) et le coût garanti plafonné à 0 €.

---


## ✅ Checkpoint de Validation du Lab

- [ ] J'ai identifié l'ID de mon projet GCP et vérifié mes rôles IAM attribués avec `gcloud` et la Console.
- [ ] J'ai vérifié par l'expérience que mon compte ne peut pas élever ses propres privilèges (Principe du Moindre Privilège).
- [ ] J'ai compris l'exigence de colocalisation géographique des données (`EU` vs `US`).
- [ ] J'ai testé l'option de sécurité `--maximum_bytes_billed` et vérifié que BigQuery stoppe automatiquement à 0 € les requêtes trop volumineuses.
- [ ] J'ai consulté les quotas d'utilisation BigQuery dans le panneau IAM & Admin.